<!-- 학습 보강 셀 -->

# 04. SimpleDirectoryReader 학습 흐름

이 노트북은 다양한 파일을 LlamaIndex `Document`로 읽는 방법을 비교합니다.
같은 폴더라도 전체 로드, 특정 파일만 로드, 특정 확장자만 로드처럼 목적에 따라 로딩 전략이 달라집니다.

<!-- 학습 보강 셀 -->

## input_files를 쓰는 상황

`input_files`는 실험 대상을 명확히 고정하고 싶을 때 유용합니다.
예를 들어 JSON과 TXT 로더 결과만 비교하거나, 특정 문서 몇 개로만 인덱스를 만들 때 사용합니다.

In [ ]:
from llama_index.core import SimpleDirectoryReader

# NewData 폴더의 여러 파일을 Document 목록으로 읽어옵니다.
# 기본 설정은 지원 가능한 파일 형식을 자동으로 읽으려고 하므로, 실습 환경에 따라 추가 패키지가 필요할 수 있습니다.
documents = SimpleDirectoryReader('../NewData').load_data()

In [ ]:
# 로드된 파일 목록 확인
# - 메타데이터의 file_name을 보면 어떤 파일이 Document로 변환되었는지 빠르게 확인할 수 있습니다.
print('디렉토리 파일 개수:', len(documents))
for document in documents:
    print(document.metadata.get('file_name'))

<!-- 학습 보강 셀 -->

## 전체 폴더 로드의 장단점

전체 폴더 로드는 빠르게 실험하기 좋지만, 원하지 않는 파일까지 함께 들어올 수 있습니다.
실제 프로젝트에서는 `.DS_Store`, 임시 파일, 이미지, 오래된 문서가 섞여 검색 품질을 떨어뜨릴 수 있으므로 필터링 전략이 필요합니다.

In [ ]:
# 특정 파일만 로드
# - input_files에는 실제로 읽고 싶은 파일 경로만 지정합니다.
documents = SimpleDirectoryReader(
    input_dir='../NewData',
    input_files=['../NewData/json_example.json', '../NewData/txt_example.txt'],
).load_data()

print('읽어온 문서 수:', len(documents))
for document in documents:
    print(document.metadata.get('file_name'))
    print(document.text[:300])
    print('-' * 80)

In [ ]:
# 특정 파일만 제외
# - exclude에 넣은 파일은 같은 디렉토리에 있어도 로드하지 않습니다.
documents = SimpleDirectoryReader(
    input_dir='../NewData',
    exclude=['../NewData/json_example.json', '../NewData/txt_example.txt'],
).load_data()

print('읽어온 문서 수:', len(documents))
for document in documents[:3]:
    print(document.metadata.get('file_name'))

<!-- 학습 보강 셀 -->

## exclude를 쓰는 상황

`exclude`는 폴더 대부분은 사용하되 일부 파일만 빼고 싶을 때 적합합니다.
데이터 폴더를 유지한 채 실험 조건만 바꿀 수 있어, 전체 로딩과 필터링 로딩의 차이를 비교하기 좋습니다.

In [ ]:
# 특정 확장자만 로드: TXT
# - required_exts는 파일 종류별 로딩 실습이나 대용량 폴더 필터링에 유용합니다.
documents = SimpleDirectoryReader(
    input_dir='../NewData',
    required_exts=['.txt'],
).load_data()

print('읽어온 문서 수:', len(documents))
print(documents[0].text)

In [ ]:
# 특정 확장자만 로드: CSV
# - CSV도 Document 텍스트로 변환되며, 표 구조는 리더 구현에 따라 문자열 형태로 표현됩니다.
documents = SimpleDirectoryReader(
    input_dir='../NewData',
    required_exts=['.csv'],
).load_data()

print('읽어온 문서 수:', len(documents))
print(documents[0].text)

<!-- 학습 보강 셀 -->

## 표 데이터 로딩 시 주의점

CSV나 Excel은 텍스트 문서와 달리 행과 열의 구조가 중요합니다.
단순 텍스트로 변환하면 표의 관계가 약해질 수 있으므로, 실제 서비스에서는 컬럼 설명이나 행 단위 변환 전략을 별도로 설계하는 것이 좋습니다.

In [ ]:
# 특정 확장자만 로드: 이미지
# - 이미지 파일은 본문 텍스트보다 파일 경로/메타데이터 확인에 초점을 둡니다.
# - PIL은 이미지를 노트북에서 열어 확인하기 위해 사용합니다.
from PIL import Image

documents = SimpleDirectoryReader(
    input_dir='../NewData',
    required_exts=['.jpg', '.jpeg', '.png'],
).load_data()

print('읽어온 이미지 문서 수:', len(documents))
image_path = documents[0].metadata['file_path']
print('이미지 경로:', image_path)
Image.open(image_path)

<!-- 학습 보강 셀 -->

## 이미지 파일과 RAG

기본 텍스트 기반 RAG에서는 이미지 자체보다 이미지 경로와 메타데이터만 다루는 경우가 많습니다.
이미지 내용을 검색하려면 OCR, 이미지 캡셔닝, 멀티모달 임베딩 같은 추가 단계가 필요합니다.